In [1]:
import time_dependent_accumulation_surface as tda
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tqdm import tqdm

d = 1.0          # nm (offset desde el origen al fondo del cilindro)
R = L = 5.0*d    # nm
D = 1.0          # nm^2 / µs
m_max = 500
p_max = 500
n_rho = 500
n_z = 500

modes = tda.precompute_modes_gauss(R, L, d, D, m_max, p_max,n_rho=n_rho,n_z=n_z)

# ----- parámetros -----
dim_factor = np.arange(0.001, 50.02, 0.1)
R_vals = dim_factor * d
L_vals = dim_factor * d

NR = len(R_vals)
NL = len(L_vals)

G_surf_inf_var = np.zeros((NR, NL))

# ----- función para evaluar una fila -----
def compute_row(i):
    R = dim_factor[i] * d
    row = np.zeros(NL)

    surface_cache = {}  # cache LOCAL por fila

    for j in range(NL):
        L = dim_factor[j] * d

        if L not in surface_cache:
            surface_cache[L] = tda.precompute_surface_integrals(
                R, L, d, modes,
                n_rho=n_rho,
                n_z=n_z
            )

        surface_ints = surface_cache[L]
        surf_modes =  tda.precompute_surface_modes(R, L, d, modes, surface_ints)
        row[j] =  tda.G_surface_infty_from_modes(modes, surf_modes)

    return i, row


In [ ]:
from joblib import Parallel, delayed
# ----- paralelización -----
results = Parallel(n_jobs=4,verbose=1)(
    delayed(compute_row)(i)
    for i in tqdm(range(NR))
)

for i, row in results:
    G_surf_inf_var[i,:] = row


In [ ]:
np.savetxt("data/G_matrix.dat", G_surf_inf_var)
np.savetxt("data/R_vals.dat", R_vals)
np.savetxt("data/L_vals.dat", L_vals)

In [ ]:
plt.imshow(G_surf_inf_var, cmap='plasma', interpolation='nearest')

# Add colorbar
plt.colorbar()

plt.title("Heatmap with color bar")
plt.show()

NameError: name 'G_surf_inf_var' is not defined